# 11 — Histology results aggregation

Mirror of notebook 06 for the LC25000 experiment. Loads per-fold JSON files, builds the final results table, ROC + PR curves, confusion matrices, bar chart, and pairwise Wilcoxon p-values.

## Why all models saturate near AUC = 1.0

The task in this notebook is **`lung_n` (benign) vs `lung_aca + lung_scc` (malignant)**. 
On LC25000 the visual gap between normal lung parenchyma and carcinoma tissue is large: cell density, nuclear morphology, and stain distribution all differ dramatically. Color-histogram features alone already separate the classes almost perfectly, which is why every model — including KNN — hits AUC ≈ 1.000.

We keep these numbers as the **upper bound** of what the pipeline can do on a visually trivial task. For an experiment that actually distinguishes models, see notebook **16 — subtype results** (lung_aca vs lung_scc), where the two malignant subtypes are visually similar and AUC drops into a realistic range.

**Caveat on sample sizes.** Classical models were trained on the full lung subset (15 000 images, 3 000 per test fold); ResNet50 was trained on the `max_per_class=1000` subsample (3 000 images, 600 per test fold). The subtype experiment uses the same subsample for both, so the comparison there is direct.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    auc,
    average_precision_score,
)

from utils.training import load_fold_results
from utils.metrics import aggregate_folds, per_fold_metric_list
from utils.stats import format_results_table, pairwise_wilcoxon

with open('../configs/histology.yaml') as f:
    cfg = yaml.safe_load(f)

results_dir = Path('..') / cfg['paths']['results']
figures_dir = Path('..') / cfg['paths']['figures']
figures_dir.mkdir(parents=True, exist_ok=True)

MODELS = ['SVM', 'RF', 'KNN', 'ResNet50_pretrained', 'ResNet50_scratch']
all_folds = {name: load_fold_results(results_dir / f'histology_{name}.json') for name in MODELS}

In [2]:
agg = {name: aggregate_folds([f.metrics_calibrated for f in folds]) for name, folds in all_folds.items()}
table = format_results_table(agg)
table.to_csv(results_dir / 'histology_summary_calibrated.csv')
table

,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,auc_roc,pr_auc
model,,,,,,,,
SVM,0.999 ± 0.001,0.999 ± 0.002,1.000 ± 0.000,1.000 ± 0.000,0.999 ± 0.001,0.999 ± 0.001,1.000 ± 0.000,1.000 ± 0.000
RF,1.000 ± 0.001,0.999 ± 0.001,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.001,1.000 ± 0.001,1.000 ± 0.000,1.000 ± 0.000
KNN,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000,1.000 ± 0.000
ResNet50_pretrained,0.998 ± 0.003,0.997 ± 0.004,1.000 ± 0.000,1.000 ± 0.000,0.999 ± 0.002,0.999 ± 0.002,1.000 ± 0.000,1.000 ± 0.000
ResNet50_scratch,0.995 ± 0.004,0.995 ± 0.006,0.996 ± 0.004,0.998 ± 0.002,0.996 ± 0.003,0.995 ± 0.004,1.000 ± 0.000,1.000 ± 0.000


In [3]:
rows = []
for name, folds in all_folds.items():
    for f in folds:
        rows.append({'model': name, 'fold': f.fold, **f.metrics_calibrated})
per_fold_df = pd.DataFrame(rows)
per_fold_df.to_csv(results_dir / 'histology_per_fold.csv', index=False)
per_fold_df

,model,fold,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,tp,tn,fp,fn,threshold,auc_roc,pr_auc
0,SVM,0,0.996667,0.9950,1.000,1.000000,0.997494,0.99750,1990,1000,0,10,0.957767,1.000000,1.000000
1,SVM,1,1.000000,1.0000,1.000,1.000000,1.000000,1.00000,2000,1000,0,0,0.948137,1.000000,1.000000
2,SVM,2,1.000000,1.0000,1.000,1.000000,1.000000,1.00000,2000,1000,0,0,0.946728,1.000000,1.000000
3,SVM,3,1.000000,1.0000,1.000,1.000000,1.000000,1.00000,2000,1000,0,0,0.938175,1.000000,1.000000
4,SVM,4,0.999667,0.9995,1.000,1.000000,0.999750,0.99975,1999,1000,0,1,0.757235,1.000000,1.000000
5,RF,0,0.998333,0.9975,1.000,1.000000,0.998748,0.99875,1995,1000,0,5,0.810000,1.000000,1.000000
6,RF,1,1.000000,1.0000,1.000,1.000000,1.000000,1.00000,2000,1000,0,0,0.830000,1.000000,1.000000
7,RF,2,0.999667,0.9995,1.000,1.000000,0.999750,0.99975,1999,1000,0,1,0.860000,1.000000,1.000000
8,RF,3,1.000000,1.0000,1.000,1.000000,1.000000,1.00000,2000,1000,0,0,0.910000,1.000000,1.000000
9,RF,4,0.999667,0.9995,1.000,1.000000,0.999750,0.99975,1999,1000,0,1,0.840000,1.000000,1.000000


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, folds in all_folds.items():
    y_true = np.concatenate([np.array(f.y_true) for f in folds])
    y_proba = np.concatenate([np.array(f.y_proba) for f in folds])
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    p, r, _ = precision_recall_curve(y_true, y_proba)
    axes[1].plot(r, p, label=f'{name} (AP={average_precision_score(y_true, y_proba):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC — out-of-fold'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR — out-of-fold'); axes[1].legend()
plt.tight_layout(); plt.savefig(figures_dir / 'histology_roc_pr.png', dpi=140); plt.show()

/var/folders/v_/vcl9p7t96w78vd6w2ddjyw2w0000gn/T/ipykernel_38355/3136586544.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(figures_dir / 'histology_roc_pr.png', dpi=140); plt.show()


In [5]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 4))
for ax, (name, folds) in zip(axes, all_folds.items()):
    y_true = np.concatenate([np.array(f.y_true) for f in folds])
    y_proba = np.concatenate([np.array(f.y_proba) for f in folds])
    thr = float(np.mean([f.threshold_calibrated for f in folds]))
    y_pred = (y_proba >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['benign', 'malignant'], yticklabels=['benign', 'malignant'])
    ax.set_title(f'{name}\nthr={thr:.2f}'); ax.set_xlabel('predicted'); ax.set_ylabel('actual')
plt.tight_layout(); plt.savefig(figures_dir / 'histology_confusion.png', dpi=140); plt.show()

/var/folders/v_/vcl9p7t96w78vd6w2ddjyw2w0000gn/T/ipykernel_38355/1791086488.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(figures_dir / 'histology_confusion.png', dpi=140); plt.show()


In [6]:
metrics = ['accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'balanced_accuracy', 'auc_roc', 'pr_auc']
means = pd.DataFrame({m: {name: agg[name][m]['mean'] for name in MODELS} for m in metrics})
stds = pd.DataFrame({m: {name: agg[name][m]['std'] for name in MODELS} for m in metrics})
ax = means.T.plot(kind='bar', yerr=stds.T, figsize=(14, 5), capsize=2)
ax.set_ylim(0, 1.05); ax.set_ylabel('score'); ax.set_title('LC25000 — model comparison (5-fold CV, calibrated)')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(figures_dir / 'histology_bars.png', dpi=140); plt.show()

/var/folders/v_/vcl9p7t96w78vd6w2ddjyw2w0000gn/T/ipykernel_38355/3435834126.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(figures_dir / 'histology_bars.png', dpi=140); plt.show()


In [7]:
auc_by_model = {name: per_fold_metric_list([f.metrics_calibrated for f in folds], 'auc_roc')
                for name, folds in all_folds.items()}
p_matrix = pairwise_wilcoxon(auc_by_model)
p_matrix.to_csv(results_dir / 'histology_wilcoxon_auc.csv')
p_matrix

,SVM,RF,KNN,ResNet50_pretrained,ResNet50_scratch
SVM,1.000,1.000,1.000,1.000,0.125
RF,1.000,1.000,1.000,1.000,0.125
KNN,1.000,1.000,1.000,1.000,0.125
ResNet50_pretrained,1.000,1.000,1.000,1.000,0.125
ResNet50_scratch,0.125,0.125,0.125,0.125,1.000
